# 07 — Predictive model: the final figures

**Figure 4** (one figure, two halves) and **Figure 5**, and the checks that go with them.
First half: Cam-CAN, cross-validated. Second half: trained on all of Cam-CAN, tested on HCP.

The slow parts (the SHAP refits, the transfer's fits) are **read from cache**: run
`sbatch slurm/predictive_cache.sbatch -- --space BSTM_ATLAS,WINDOW_S,BSTM_STATES --model MODEL --arm BSTM_ARM` once
(the values of the Parameters cell); a part not cached is computed here, once, and cached.

| plot | was | what |
|---|---|---|
| Figure 4, left | 10.9 | bstm vs static FC, Cam-CAN (80% / 20%), what the bstm model uses |
| 7.1 | 10.7 | volcano, anxiety / depression vs Normal |
| Figure 5a, 5b | 10.12 | where the model looks, Cam-CAN: cross-validated SHAP; SHAP of the model fitted on all of Cam-CAN |
| 7.2, 7.3 | 10.10, 10.11 | observed vs out-of-fold prediction; stability over fold seeds |
| Figure 4, right | 13.4 | bstm vs static FC, Cam-CAN → HCP |
| 7.4 | 13.5 | predicted vs observed, HCP |


## Parameters

In [1]:
# ---- THE MODELS OF THIS NOTEBOOK (both halves: Cam-CAN cross-validated and Cam-CAN -> HCP)
BSTM_ATLAS  = "yeo7"     # brain-state transition model: the state set's atlas
BSTM_STATES = "ThresholdCluster_umap3_8"      #   ... and its states
WINDOW_S    = -1               #   ... and its window (s)
BSTM_ARM    = "all [joint]"        #   ... and its variables (arm)
FC_ATLAS    = "yeo7"              # static FC: atlas
FC_ARM      = "edges"             #   ... and its variables ("edges" | "global")
MODEL       = "hgb"               # both: "hgb" | "ridge"

# The SHAP fits are read from the cache that slurm/predictive_cache.sbatch writes, never fitted here
# (the notebook image has no `shap`). After changing a parameter above, run once:
print(f'sbatch slurm/predictive_cache.sbatch -- --space {BSTM_ATLAS},{WINDOW_S:g},{BSTM_STATES} '
      f'--model {MODEL} --arm "{BSTM_ARM}" --fc-arm {FC_ARM}')
# ---- how the figures name the states: "number" (S3) | "name" | "both" (S3 · name); the names come
# from notebook 04 (state_names), per state set -- read in the setup cell below
STATE_LABELS = "name"


sbatch slurm/predictive_cache.sbatch -- --space yeo7,-1,ThresholdCluster_umap3_8 --model hgb --arm "all [joint]" --fc-arm edges


## Part 1 — Cam-CAN, cross-validated

In [2]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# every figure of a "# Plot N.k" / "# Figure N" cell is written to notebooks/figures/07/ (N_k.png), overwritten each run
from fmri_decomposition.figures import autosave
autosave.enable(REPO / "notebooks" / "figures" / "07")

# the states' names (notebook 04) for every figure, as STATE_LABELS says
from fmri_decomposition.io import default_output_root
from fmri_decomposition.figures import state_names as sn
sn.use_for(default_output_root(), BSTM_ATLAS, WINDOW_S, BSTM_STATES, None, mode=STATE_LABELS)

from fmri_decomposition.supervised_model import hads as ht
from fmri_decomposition.supervised_model import report as rp
from fmri_decomposition.supervised_model import transfer_tables as tt
from fmri_decomposition.figures import hads as hp

# Cam-CAN, notebook 10's settings (report.CAM); override any of them here, e.g. save_dir="figures/07"
cfg = rp.cam_settings()
st = ht.Study(cfg)
st.cache_only = True               # the slow fits come from slurm/predictive_cache.sbatch
T0, T1 = cfg.tasks
print(f"phenotype: {len(st.pheno):,} subjects;",
      ", ".join(f"{lab} {st.pheno[lab].value_counts().sort_index().astype(int).to_dict()}" for lab in cfg.targets))

autosave: figures of '# Plot N.k' / '# Figure N' cells -> /project/6008063/tamires/DecomposingfMRI/notebooks/figures/07
state labels: name · S0 Social Cognition, S1 Language & Music, S2 Cognitive Attention, S3 Somethesis & Action Execution, S4 Explicity Memory & Reasoning, S5 Emotion, S6 Vision, S7 Somethesis & Soma Cognition
phenotype: 2,681 subjects; anxiety {0.0: 2053, 1.0: 362, 2.0: 179, 3.0: 21}, depression {0.0: 2369, 1.0: 161, 2.0: 67, 3.0: 7}


In [3]:
SPACE = dict(atlas=BSTM_ATLAS, window_s=WINDOW_S, states=BSTM_STATES)
base, options, space, model = rp.cam_choices(st, SPACE, dict(model=MODEL, arm=BSTM_ARM))
label = f"{model['model']} · {model['arm']} · {ht.set_label(space)}"
print("space:", ht.set_label(space), "| model:", label)


baselines: read /project/6008063/tamires/DecomposingfMRI/outputs/modelling/version=v0/supervised_model/cache/baselines_2c2d87de45cd.parquet
space: ThresholdCluster_umap3_8 · yeo7 · 1 TR | model: hgb · all [joint] · ThresholdCluster_umap3_8 · yeo7 · 1 TR


### Figure 4, left half (was 10.9)

Drawn here, shown with the right half at the end.

In [4]:
# Figure 4, left half (was 10.9)
SAVE_AS = None    # None = "figure_4_left_half" -> notebooks/figures/07/figure_4_left_half.png; to keep another version: SAVE_AS = "figure_4_left_half_version2"
perf = st.performance_trees(space, model["arm"], MODEL, base, fc_arm=FC_ARM, fc_atlas=FC_ATLAS)
e1_seeds = st.tree_seeds(space, model["arm"], MODEL, fc_arm=FC_ARM, fc_atlas=FC_ATLAS)
deep = st.deep(space, model)           # out-of-fold SHAP; cached by slurm/predictive_cache.sbatch
e1_shares = st.block_shares(deep)
fc_note = "" if (perf["tree"] == "fcm").any() else f" -- no fcm run on {FC_ATLAS}"
fig4_left = hp.fig_performance(perf, cfg, "",
                               {"bstm": f"{ht.set_label(space)} · {model['arm']}",
                                "fcm": f"{FC_ATLAS} · same model · {FC_ARM}{fc_note}"},
                               MODEL, seeds=e1_seeds, shares=e1_shares, show=False, baseline=False)

SystemExit: deep is not cached for this choice (('yeo7', -1.0, 'ThresholdCluster_umap3_8', 'hgb', 'all [joint]')...) and this Study is cache_only: it is fitted by the cluster job, not in a notebook kernel (which may not even have `shap`). Run, with the notebook's parameters:
  sbatch slurm/predictive_cache.sbatch -- --space ATLAS,WINDOW_S,STATES --model MODEL --arm "ARM"
then rerun this cell.

/opt/venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### Plot 7.1 — anxiety and depression vs Normal (was 10.7)

In [ ]:
# Plot 7.1
SAVE_AS = None    # None = "7_1" -> notebooks/figures/07/7_1.png; to keep another version: SAVE_AS = "7_1_version2"
grp = st.group_tests(space)
hp.fig_volcanos(grp, cfg, f"{cfg.case_from}+ vs Normal (on both scales), adjusted · {ht.set_label(space)}",
                "D2_volcano_groups", f"log2 fold change  ({cfg.case_from}+ vs Normal, adjusted, shrink={cfg.shrink:g})",
                alpha=0.10, fc_line=0.25, xlim=0)

### Figure 5 — where the model looks, Cam-CAN (was 10.12)

Red = a variable that passes **both** criteria: the model relies on it (among its top `highlight_top`
by mean |SHAP|, in the top for ≥ `stable_min` of the seeds) **and** the group differs from Normal
(|Cohen's d| ≥ `min_d`, ≥ 1 pp). Two versions, the same groups and graphs, two SHAPs:

* **5a, cross-validated**: out-of-fold SHAP of the Cam-CAN model (5 folds × seeds), the left half of Figure 4.
* **5b, the transfer's model**: the model fitted on **all** of Cam-CAN, the one tested on HCP (right half
  of Figure 4), its SHAP on the Cam-CAN subjects (one fit per seed).

In [ ]:
# Figure 5a
SAVE_AS = None    # None = "figure_5a" -> notebooks/figures/07/figure_5a.png; to keep another version: SAVE_AS = "figure_5a_version2"
paired = st.paired(space)              # movie vs rest, Normal group (the dynamics beside the graphs)
grp = st.group_tests(space) if "grp" not in globals() else grp
pooled = st.pooled(space)
dyn = st.dynamics_marks(space, paired, grp)
st.check(deep)
cfg.change_unit = "percent"   # labels on the graphs: "percent" (relative: +2 pp on .10 = +20%) | "pp" (points of probability)
SAME_GREY = True   # True: the depression / anxiety panels in Normal's grey | False: lighter grey
refs_model = {(t, lab): (t, "Normal") for t in cfg.tasks for lab in cfg.targets}
rule = (f"red = the model's top {cfg.highlight_top} graph variables by mean |SHAP|, top in "
        f"≥{cfg.stable_min:.0%} of seeds, |Cohen's d| ≥ {cfg.min_d} vs Normal; label = "
        + ("change relative to Normal (%)" if cfg.change_unit == "percent" else "change in pp"))
mmarks = st.model_marks(space, deep, pooled)
hp.fig_graph(pooled, mmarks, cfg, refs_model, {k: hp.RED for k in refs_model},
             f"Figure 5a  cross-validated SHAP · {label}", [(hp.RED, rule)], "figure5a_graph", dyn=dyn.assign(hit=False),
             grey_compared=not SAME_GREY)

In [ ]:
# Figure 5b
SAVE_AS = None    # None = "figure_5b" -> notebooks/figures/07/figure_5b.png; to keep another version: SAVE_AS = "figure_5b_version2"
full = st.full(space, model)           # fitted on all of Cam-CAN (the transfer's model); cached
fmarks = st.model_marks(space, full, pooled)
hp.fig_graph(pooled, fmarks, cfg, refs_model, {k: hp.RED for k in refs_model},
             f"Figure 5b  SHAP of the model fitted on all of Cam-CAN (tested on HCP) · {label}",
             [(hp.RED, rule)], "figure5b_graph", dyn=dyn.assign(hit=False),
             grey_compared=not SAME_GREY)

### Plots 7.2, 7.3 — prediction and its stability (were 10.10, 10.11)

In [ ]:
# Plots 7.2, 7.3
hp.fig_predicted(deep, cfg, f"observed category vs out-of-fold prediction · {label}")
hp.fig_stability(deep, cfg, f"stability over fold seeds · {label}")


## Part 2 — trained on all of Cam-CAN, tested on HCP

In [ ]:
# Cam-CAN -> HCP, notebook 13's setup (report.transfer_setup)
TARGET = "category"      # "raw" | "category" (the manuscript)
train, test, train_cfg, test_cfg, ROOT = rp.transfer_setup(TARGET)
train.cache_only = True            # the transfer's SHAP and predictions come from the sbatch
raw_rows, rows_t = rp.transferred_rows(ROOT, TARGET)
print(f"{len(rows_t)} transferred configuration(s) over {rows_t['states'].nunique()} state set(s); rows from {ROOT}")


In [ ]:
# the configuration of the parameters, in the transferred rows
PICK = dict(atlas=BSTM_ATLAS, window_s=WINDOW_S, states=BSTM_STATES, model=MODEL)
# The transfer runs its arms without a suffix, cells as stored = p(i, j): its "all" IS "all [joint]".
have = set(rows_t["arm"])
base, _, sfx = BSTM_ARM.partition(" [")
arm_t = BSTM_ARM if BSTM_ARM in have else base if sfx == "joint]" and base in have else None
if arm_t is None:
    raise SystemExit(f"BSTM_ARM={BSTM_ARM!r} has no transfer: the transferred arms are {sorted(have)} "
                     f"(\"all\" = \"all [joint]\"). Set BSTM_ARM = \"all [joint]\", or run the transfer in [cond].")
PICK["arm"] = arm_t
best, ranked, n_clear = rp.choose_best(rows_t, PICK, 0)
space_t = rp._space(best)
display(ranked.head(8)[["states", "atlas", "window", "arm", "model", "task", "phenotype", "score", "lo", "hi", "base_score"]])


### Figure 4, right half (was 13.4)

In [ ]:
# Figure 4, right half (was 13.4)
SAVE_AS = None    # None = "figure_4_right_half" -> notebooks/figures/07/figure_4_right_half.png; to keep another version: SAVE_AS = "figure_4_right_half_version2"
perf_t = tt.transfer_perf(rows_t, best, fc_arm=FC_ARM, fc_atlas=FC_ATLAS)
shares_t = rp.transfer_shares_cached(train, test, space_t, best["arm"], best["model"])   # cached
fig4_right = hp.fig_performance(
    perf_t, train_cfg, "",
    {"bstm": f"{best['label']} · {best['arm']}", "fcm": f"{FC_ATLAS} · {FC_ARM}"},
    best["model"], shares=shares_t,
    share_label="share of mean |SHAP| on the TEST subjects (one fit on all of Cam-CAN, no folds)",
    show=False, baseline=False)

### Plot 7.4 — predicted vs observed, HCP (was 13.5)

In [ ]:
# Plot 7.4
SAVE_AS = None    # None = "7_4" -> notebooks/figures/07/7_4.png; to keep another version: SAVE_AS = "7_4_version2"
pred = rp.transfer_predictions_cached(train, test, best, fc_arm=FC_ARM)   # cached
hp.fig_scatter(pred, train_cfg, title=f"predicted vs observed · {best['label']} · {best['model']}")

## Figure 4

In [ ]:
# Figure 4
SAVE_AS = None    # None = "figure_4" -> notebooks/figures/07/figure_4.png; to keep another version: SAVE_AS = "figure_4_version2"
hp.fig_figure4(fig4_left, fig4_right, cfg,
               titles=("Train in 80% Cam-CAN → test in 20% Cam-CAN",
                       "Train in 100% Cam-CAN → test in 100% HCP"));